# Lecture 3 follow-along — Python That Behaves Like a Robot

Every code cell is **verbatim** from the lecture (hidden lecture helpers are included and marked). In class these were `lec3_warmup.py`, `lec3_sensor.py`, `lec3_filter.py`, run with `python3 <file>`. Run cells top to bottom.

*(down = deeper on every depth plot, as always.)*

## Warm-up (Week 0 muscle) — predict the output first

In [ ]:
readings = [1.2, 2.6, 0.4, 2.9, 1.1]   # depth samples [m]
count = 0                               # the tally — born at ZERO, nothing counted yet
for r in readings:                      # visit each reading in turn
    if r > 2.0:                         # deeper than 2 m?
        count += 1                      # count it (= count + 1)
print(count, "readings over 2 m")

## Build File 1 — `lec3_sensor.py`: the lying sensor
$z_k = d_{true} + \varepsilon_k$, seed 7 (the scramble machine).

In [ ]:
import numpy as np                    # numerical toolbox, nicknamed np
rng = np.random.default_rng(7)        # lie factory — seed 7: same lies every run

t = np.arange(0, 20, 0.1)             # t: 0.0, 0.1, ... 19.9 (20 s at 10 Hz)
true_depth = 2.0 + 0.0*t              # d_true — flat 2 m, one value per tick
z = true_depth + rng.normal(0, 0.08, t.size)   # z_k = d_true + ε_k
print("first five readings:", np.round(z[:5], 3))

## See the lie

In [ ]:
import matplotlib.pyplot as plt                            # plotting library, nicknamed plt
plt.plot(t, z, '.', ms=3, label='sensor $z_k$')            # '.' dots, markersize 3
plt.plot(t, true_depth, 'k--', label='truth $d_{true}$')   # 'k--' = black, dashed
plt.gca().invert_yaxis()                                   # COURSE RULE: deeper = lower on page
plt.xlabel('time [s]'); plt.ylabel('depth [m] (down = deeper)'); plt.legend(fontsize=8)
plt.savefig("lie.png"); plt.show()   # savefig: the container has no pop-up windows — plots become FILES

*(lecture helper, shown here so plots below run)*

In [ ]:
def depth_axis():
    """The three repeated axis lines, bottled once (DRY)."""
    plt.gca().invert_yaxis()
    plt.xlabel('time [s]'); plt.ylabel('depth [m] (down = deeper)')
    plt.legend(fontsize=8)

## Build File 2 — `lec3_filter.py`: the moving average
Toy test first: predict `[1, 1, 1, 10, 1, 1]` with N=3, then run.

In [ ]:
def moving_average(z, N):
    out, window = [], []
    for zi in z:                    # the conveyor belt, from the walkthrough
        window.append(zi)
        if len(window) > N:
            window.pop(0)
        out.append(sum(window)/len(window))
    return out

test = [1, 1, 1, 10, 1, 1]          # one wild spike in calm readings
print(moving_average(test, 3))      # predict first, then run

## Does it work? (uses your sensor data from above)

In [ ]:
# needs the sensor data too — copy your 6 lines from lec3_sensor.py up top
plt.plot(t, z, '.', ms=3, alpha=0.4, label='raw')           # alpha: 40% opaque
plt.plot(t, moving_average(z, 10), lw=2, label='filtered N=10')  # lw: line width
plt.plot(t, true_depth, 'k--', label='truth')
depth_axis()   # our three repeated axis lines, bottled once
plt.savefig("filter.png"); plt.show()

## No free lunch — smoothness buys lag

In [ ]:
true2 = np.where(t < 8, 1.0, 2.0)    # np.where(cond, a, b): array-wide if/else
z2 = true2 + rng.normal(0, 0.08, t.size)

plt.plot(t, z2, '.', ms=3, alpha=0.35, label='raw')
plt.plot(t, moving_average(z2, 10), lw=2, label='N=10')
plt.plot(t, moving_average(z2, 40), lw=2, label='N=40')
plt.plot(t, true2, 'k--', label='truth')
plt.annotate('N=40 still climbing —\nreporting the PAST', xy=(10.3, 1.72), xytext=(12.3, 1.15),
    fontsize=8, color='#9a5400', arrowprops=dict(arrowstyle='->', color='#9a5400'))
depth_axis()                         # annotate: arrow FROM the words TO xy
plt.show()

## The depth alarm — the robot that *notices*
Theory: lag ≈ N/2 × 0.1 s → N=10 fires 0.5 s late, N=40 ≈ 2 s late.

In [ ]:
def depth_alarm(t, filtered, limit):
    for i in range(len(filtered)):   # walk the ticks in order
        if filtered[i] > limit:      # deeper than allowed? (down = bigger)
            return t[i]              # return the TIME — and stop: FIRST crossing
    return None                      # loop survived: never crossed

print("N=10 alarm at t =", depth_alarm(t, moving_average(z2, 10), 1.5))
print("N=40 alarm at t =", depth_alarm(t, moving_average(z2, 40), 1.5))